### The governing equation of a simple pendulum is : $$\ddot{\theta} + \frac{g}{L} sin(\theta) + b\ddot{\theta} = 0$$

Where, 
$$\theta = angular\  \  displacement$$
$$\dot{\theta} = angular\  \ velocityd$$
$$\ddot{\theta} = angular\  \  acceleration$$
$$b = damping\ \  constant$$
$$g = acceleration\  \  due\ \ to\ \ gravity%$$
$$L  = length \ \ of\ string$$

#### To solve the above differential equation we use `solve_ivp tool` form `scipy.integrate module`

#### Formulation of above differential equation for numerical solution: 
We choose another variable `omega` such that :$$\frac{d\dot{\theta}}{dt} = \omega$$
Now, 
$$\frac{d\dot{\theta}}{dt} = \omega$$
$$\frac{d\dot{\omega}}{dt} = \ddot{\theta} = - \frac{g}{L}\ \sin(\theta) - b \omega$$

For using interactive plotting in vs code I used `Qt5Agg` backend of matplotlib

In [4]:
import matplotlib
#matplotlib.use('TkAgg') 
matplotlib.use('Qt5Agg')

Importing all the essential packages

In [5]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from matplotlib.animation import FuncAnimation, PillowWriter, FFMpegWriter

Initialinzing time steps and differential function to pass into `solve_ivp`

In [7]:
t = np.linspace(0,15, 1000)

def pendulum_dynamics(t,x):
    g = 9.81
    theta, thetadot = x
    return thetadot, (-g/5)*np.sin(theta) -  0.7 *thetadot

y0= (2, 2)

`solve_ivp(fun, t_span, y0, method='RK45', t_eval=None)`
Here,
- y0 is initial condition,
- t_span is the interval of t in which we are solving the differential equation.
- t_eval argument specifies the time points at which you want the solution to be evaluated and returned.


In [9]:
sol = solve_ivp(pendulum_dynamics, (0,15), y0, t_eval = t, max_step= 0.01, rtol = 1e-9, atol = 1e-12)

Defning text earlier. This makes the code cleaner.

In [12]:
gov_eqn = r'$\ddot{\theta} + \frac{g}{L} sin(\theta) + b\ddot{\theta}$'

Setting up the Canvas for animation.

In [ ]:
## Here , Iam setting two subplots one for the pendulum swinging animation and another for phase portrait

fig, axes = plt.subplots(1, 2, figsize = (8,4), layout = 'tight')

## Pendulum swinging subplot
ax = axes[0]
ax.set_aspect('equal', 'box')
pendulum_pivot = ax.plot(xp(r.min(), theta), yp(r.min(), theta), 'o') ## Initializing pendulum pivot about which it swings
pendulum_string, = ax.plot([],[])               ## Initializing string of the pendulum
pendulum_mass, = ax.plot([],[], 'o', color = 'red', ms = 16) ## Initializing the Bob of pendulum

ax.set_xlim(-7.5, 7.5) ## Make sure to set limits otherwise the animation will be off from canvas
ax.set_ylim(-7.5, 8)
ax.text(0.5,0.9, gov_eqn, transform = ax.transAxes,\
        bbox = {'facecolor' : 'white', 'edgecolor' : 'black'}, ha = 'center') ## Adding text for displaying governing equation

angle = ax.text(0,0, '', transform = ax.transAxes,\
        bbox = {'facecolor' : 'white', 'edgecolor' : 'black'}, ha = 'center') ## Adding text for displaying angular displacement of the pendulum
ax.tick_params(axis = 'both', labelsize = 8) ## Modifying the ticks parameter to make tick size smaller
ax.set_title('Simple Pendulum')
ax.grid(True, ls = '--', lw = 0.7)

## Initializin initial conditions to make phase portrait
ics = [[0.174, 0],[2.617, 3],[3.159, 1.5],[3.31, 0],[0, 1],[0,2.8],[0, 1.5],[4, -2],[2.617, 1.1],[2, 2],[2.54, -3]] 


## Phase portrait subplot
ax = axes[1]
ax.set_aspect('equal', 'box')
for ic in ics:                          ## To make phase portrait we have to solve the differential equation imposing the initial conditions using loop
    sols = solve_ivp(pendulum_dynamics, (0,15), ic, t_eval= t)
    plt.plot(sols.y[0], sols.y[1])

phase_point, = ax.plot([],[], 'b-o', ms = 8, label = 'State of The System') ## Initializing a point to loacate the state of system in phase portrait
state = ax.text(0,0, '', transform = ax.transAxes, \
                 bbox = {'facecolor' : 'white', 'edgecolor' : 'black'}) ## Initializing text to display state of the system


ax.tick_params(axis = 'both', labelsize = 8)


#####  THIS SECTION IS USED TO MAKE CUSTOM TICKS IN X AXIS IN FORM OF PI

# Defining tick positions to match the labels
tick_positions = [0, np.pi/4, np.pi/2, 3*np.pi/4, np.pi, 
                  5*np.pi/4, 3*np.pi/2, 2*np.pi,
                  9*np.pi/4, 5*np.pi/2]

## Defingin tick labels to put on the above positions
tick_labels = [r'0', r'$\frac{\pi}{4}$', r'$\frac{\pi}{2}$', r'$\frac{3\pi}{4}$',
               r'$\pi$', r'$\frac{5\pi}{4}$', r'$\frac{3\pi}{2}$', r'$2\pi$',
               r'$\frac{9\pi}{4}$', r'$\frac{5\pi}{2}$']

x = ax.set_xticks(tick_positions)
y = ax.set_xticklabels(tick_labels)
##      TICKS SECTION END

ax.set_title('Phase Portrait')
ax.set_xlabel(r'Angular Displacement, $\theta$ [rad]', fontsize = 8)
ax.set_ylabel(r'Angular velocity, $\omega$ [rad/s]', fontsize = 8)
ax.grid(True, ls = '--', lw = 0.7, color = 'pink', alpha = 0.7)



### Defining animation function

In [24]:
### It takes in frame as argument and all the images are made based on which frame we are in 
def animate(frame):

    ### Defining essential data to be updated in animation
    y_0 = sol.y[0][frame] 
    y_1 = sol.y[1][frame]
  
    theta = sol.y[0][frame]
    x_mass = 5 *np.sin(theta)
    y_mass = - 5 * np.cos(theta)
    theta_pi = theta/np.pi

## Initializing text earlier to update the text
    state_text = '\n'.join((' = '.join((r'$\theta$', '% .2f'%theta)),' = '.join((r'$\omega$', '{:.2f}'.format(sol.y[1][frame])))))
    angle_txt = ' = '.join((r'$\theta$', '%.2f'%theta_pi))

## Updating the data for subplot 1
    pendulum_mass.set_data([x_mass], [y_mass])
    pendulum_string.set_data([0, x_mass], [0, y_mass])
    phase_point.set_data([y_0],[y_1])
    angle.set_text(angle_txt + r'$\pi$')
    angle.set_position((0.85, 0.05))

## Updating the data for subplot 2
    state.set_text(state_text)
    state.set_position((0.78, 0.1))
    
## Returning the updated data to form next image
    return pendulum_mass, pendulum_string, phase_point, state, angle


In [ ]:
## Initializing ani object form class FuncAnimation
ani = FuncAnimation(fig, animate, frames = 1000, interval = 2, blit = True, repeat = False)
plt.show()

#### To save the plot we have to clear the `plt.show()` section from above and then execute the line below:

Otherwise it may show static plot when it is saved after `plt.show()`